# Kangani evaluation analysis

**Two datasets. They are never blended.**

| | source | what it can support |
|---|---|---|
| **A — synthetic** | `eval/runs/*/results.jsonl`, generated by `run.py` against seeded fixtures | coverage, dispatch accuracy, branch reachability, variance |
| **B — real** | `turns` / `tool_calls` from the live Railway database | correctness, and only on hand-labelled rows |

Every table and every chart below is labelled A or B in its title. Nothing
averages across the two. Synthetic prompts were written to reach specific
branches, so a tool's synthetic frequency says how the suite was designed,
not how the bot is used.

Every number traces to a run artifact. `manifest.json` in each run directory
records the model, the effective `MAX_TOOL_ITERATIONS`, cache hit/miss
counts, and the SHA-256 of the fixture the run executed against.

In [ ]:
import json
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

EVAL_DIR = Path.cwd() if Path.cwd().name == "eval" else Path.cwd() / "eval"
REPO_ROOT = EVAL_DIR.parent
FIG_DIR = EVAL_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 120, "figure.autolayout": True,
                     "axes.grid": True, "grid.alpha": 0.25})

# The full registry, read from the source rather than typed out, so a tool
# added tomorrow shows up here as dead rather than as absent.
import tools as kangani_tools  # noqa: E402

ALL_TOOLS = sorted(kangani_tools.TOOL_HANDLERS)
print(f"{len(ALL_TOOLS)} tools registered")

## Load dataset A (synthetic)

In [ ]:
def load_runs(runs_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One row per execution, plus one row per tool call.

    Executions and tool calls are kept as separate frames rather than one
    exploded frame: a button-routed execution has zero tool calls, and
    exploding would either drop it or invent a null row for it. Both are wrong
    for a share-of-total denominator.
    """
    exec_rows, call_rows = [], []
    for run_dir in sorted(runs_dir.glob("*/")):
        results = run_dir / "results.jsonl"
        manifest_path = run_dir / "manifest.json"
        if not results.exists():
            continue
        manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
        for line in results.read_text(encoding="utf-8").splitlines():
            rec = json.loads(line)
            turns = rec.get("instrumentation", {}).get("turns") or []
            turn = turns[0] if turns else {}
            exec_rows.append({
                "run": run_dir.name,
                "state": rec["state"],
                "ceiling_config": manifest.get("ceiling_config", False),
                "max_tool_iterations": manifest.get("max_tool_iterations"),
                "prompt_id": rec["prompt_id"],
                "repeat": rec["repeat"],
                "message": rec["message"],
                "message_chars": len(rec["message"]),
                "expected_tool": rec["expected_tool"],
                "expected_route": rec["expected_route"],
                "actual_route": rec["actual_route"],
                "tool_hit": rec["tool_hit"],
                "route_hit": rec["route_hit"],
                "n_tools": len(rec["actual_tools"]),
                "iterations": turn.get("iterations", 0),
                "api_calls": turn.get("api_calls", 0),
                "api_ms": turn.get("api_ms", np.nan),
                "error": rec.get("error"),
            })
            for call in rec.get("instrumentation", {}).get("tool_calls") or []:
                call_rows.append({
                    "run": run_dir.name, "state": rec["state"],
                    "ceiling_config": manifest.get("ceiling_config", False),
                    "prompt_id": rec["prompt_id"], "repeat": rec["repeat"],
                    "tool_name": call["tool_name"],
                    "iteration_number": call["iteration_number"],
                    "latency_ms": call["latency_ms"],
                    "is_error": bool(call["is_error"]),
                })
    return pd.DataFrame(exec_rows), pd.DataFrame(call_rows)


syn_exec, syn_calls = load_runs(EVAL_DIR / "runs")
if syn_exec.empty:
    raise SystemExit("No runs found. Run eval/run.py first.")

# The default-config runs are the baseline; the lowered-ceiling runs are a
# separate condition and are held out of every baseline figure.
base = syn_exec[~syn_exec["ceiling_config"]]
base_calls = syn_calls[~syn_calls["ceiling_config"]]

print(f"[A] {len(syn_exec)} executions across {syn_exec['run'].nunique()} runs")
print(f"[A] {len(syn_calls)} tool calls")
print(syn_exec.groupby(["state", "ceiling_config"]).size().rename("executions"))

## Load dataset B (real), if it exists

Dataset B accumulates only while the instrumented bot is running. Until there
is enough of it, the sections that depend on it are skipped explicitly rather
than filled with synthetic rows.

Point `REAL_DB` at a copy of the production database.

In [ ]:
REAL_DB = EVAL_DIR / "real" / "kangani.db"

def load_real(db_path: Path):
    if not db_path.exists():
        return None, None
    conn = sqlite3.connect(db_path)
    try:
        turns = pd.read_sql_query("SELECT * FROM turns", conn)
        calls = pd.read_sql_query("SELECT * FROM tool_calls", conn)
    finally:
        conn.close()
    return turns, calls


real_turns, real_calls = load_real(REAL_DB)
HAS_REAL = real_turns is not None and len(real_turns) > 0
if HAS_REAL:
    print(f"[B] {len(real_turns)} real turns, {len(real_calls)} real tool calls")
    print(real_turns["terminated_by"].value_counts().rename("turns"))
else:
    print(f"[B] no real data at {REAL_DB} — dataset B sections will be skipped.")

## 1. Tool coverage and dead tools (A)

A tool is *dead* here in the weak sense: never invoked across the whole
synthetic suite. Since the suite contains a prompt written specifically for
every registered tool, a tool that is still never called was not reached even
when the prompt was aimed directly at it — which points at the tool
description, not at coverage.

In [ ]:
called = base_calls["tool_name"].value_counts()
coverage = (
    pd.DataFrame({"tool": ALL_TOOLS})
    .assign(calls=lambda d: d["tool"].map(called).fillna(0).astype(int))
    .sort_values("calls", ascending=False)
    .reset_index(drop=True)
)
dead = coverage.loc[coverage["calls"] == 0, "tool"].tolist()
print(f"[A] {len(ALL_TOOLS) - len(dead)}/{len(ALL_TOOLS)} tools invoked; "
      f"{len(dead)} never called")
print("dead:", dead or "none")

fig, ax = plt.subplots(figsize=(8, 9))
colors = ["#b0b0b0" if c == 0 else "#3b6ea5" for c in coverage["calls"]]
ax.barh(coverage["tool"], coverage["calls"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("tool calls (synthetic suite, default config)")
ax.set_title("[A] Tool invocation frequency — grey = never called")
fig.savefig(FIG_DIR / "01_tool_frequency.png", bbox_inches="tight")
plt.show()

## 2. Termination paths, iterations, ceiling and button share (A)

The denominator is every execution, including the ones that never reached
Claude. That is the whole reason `turns` records button routes: a rate
computed only over tool-using turns silently flatters itself.

In [ ]:
route_mix = (
    base.groupby(["state", "actual_route"]).size().unstack(fill_value=0)
)
route_share = route_mix.div(route_mix.sum(axis=1), axis=0)
print("[A] termination path share by fixture state")
print((route_share * 100).round(1))

ceiling_runs = syn_exec[syn_exec["ceiling_config"]]
if not ceiling_runs.empty:
    llm_only = ceiling_runs[ceiling_runs["actual_route"] != "button_route"]
    rate = (llm_only["actual_route"] == "ceiling").mean()
    cap = ceiling_runs["max_tool_iterations"].iloc[0]
    n_ceiling = int((llm_only["actual_route"] == "ceiling").sum())
    print(f"\n[A] ceiling config (MAX_TOOL_ITERATIONS={cap}): "
          f"{rate:.1%} of LLM turns hit the ceiling "
          f"({n_ceiling}/{len(llm_only)})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
route_share.plot(kind="bar", stacked=True, ax=axes[0], colormap="tab20")
axes[0].set_ylabel("share of executions")
axes[0].set_title("[A] Termination path by fixture state")
axes[0].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))
axes[0].tick_params(axis="x", rotation=0)

llm = base[base["actual_route"] != "button_route"]
bins = np.arange(0.5, llm["iterations"].max() + 1.5)
axes[1].hist(llm["iterations"], bins=bins, color="#3b6ea5", edgecolor="white")
axes[1].set_xlabel("tool-call rounds per turn")
axes[1].set_ylabel("executions")
axes[1].set_title("[A] Iterations per LLM turn (default config)")
fig.savefig(FIG_DIR / "02_routes_and_iterations.png", bbox_inches="tight")
plt.show()

## 3. Latency by tool (A)

`latency_ms` times `execute_tool` only — the handler, not the API round trip.
Those are stored separately (`turns.api_ms`) because almost every handler is
sub-millisecond SQLite and would otherwise vanish under ~2 s of network.

Read with care: these are local-fixture timings on a small corpus, not
production latencies.

In [ ]:
lat = (
    base_calls.groupby("tool_name")["latency_ms"]
    .agg(["count", "median", "mean", "max"])
    .sort_values("median", ascending=False)
)
print("[A] handler latency, milliseconds")
print(lat.round(3).head(15))

top = lat.head(12).index.tolist()
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(
    [base_calls.loc[base_calls["tool_name"] == t, "latency_ms"] for t in top],
    labels=top, vert=False, showfliers=False,
)
ax.set_xlabel("handler latency (ms, log scale)")
ax.set_xscale("log")
ax.set_title("[A] Tool handler latency — excludes API round trip")
fig.savefig(FIG_DIR / "03_latency_by_tool.png", bbox_inches="tight")
plt.show()

median_api = base["api_ms"].median()
print(f"\n[A] median API time per turn: {median_api:.0f} ms — "
      f"{median_api / max(lat['median'].median(), 1e-9):.0f}x the median handler")

## 4. Dispatch variance across repeats (A)

Identical input, repeated. Every repeat runs against a freshly restored
fixture, so drift is excluded by construction and what remains is model
nondeterminism.

Measured as the share of prompts whose chosen tool set was *not* identical
across all repeats.

In [ ]:
per_exec_tools = (
    base_calls.groupby(["state", "prompt_id", "repeat"])["tool_name"]
    .apply(lambda s: tuple(sorted(set(s))))
    .rename("toolset").reset_index()
)
variance = (
    per_exec_tools.groupby(["state", "prompt_id"])["toolset"]
    .agg(distinct="nunique", repeats="size").reset_index()
)
variance["unstable"] = variance["distinct"] > 1
print(f"[A] {variance['unstable'].sum()}/{len(variance)} prompts chose a "
      f"different tool set across repeats "
      f"({variance['unstable'].mean():.1%})")

by_expected = (
    base[base["expected_tool"].notna()]
    .groupby("expected_tool")["tool_hit"]
    .agg(["mean", "size"]).sort_values("mean")
)
by_expected = by_expected[by_expected["size"] >= 3]
print("\n[A] dispatch accuracy by expected tool (lowest 12)")
print(by_expected.head(12).round(3))

fig, ax = plt.subplots(figsize=(8, 8))
lowest = by_expected.head(25)
ax.barh(lowest.index, lowest["mean"], color="#c25b56")
ax.axvline(1.0, color="#444", lw=1, ls="--")
ax.set_xlim(0, 1.05)
ax.set_xlabel("share of repeats where the expected tool was called")
ax.set_title("[A] Dispatch accuracy by expected tool")
ax.invert_yaxis()
fig.savefig(FIG_DIR / "04_dispatch_accuracy.png", bbox_inches="tight")
plt.show()

## 5. Is the deterministic shortcut ever wrong? (A)

Two failure directions, and they are not symmetric.

**Over-routing** — a prompt with real scope gets short-circuited, so the
filter is silently dropped and the user gets a plausible-looking wrong
answer. This is the expensive one.

**Under-routing** — a bare canonical phrase reaches Claude anyway. Costs an
API call and nothing else.

In [ ]:
routed = base["actual_route"] == "button_route"
should_route = base["expected_route"] == "button_route"
over = base[routed & ~should_route]
under = base[~routed & should_route]

print(f"[A] over-routed (scope silently dropped): {len(over)}")
print(f"[A] under-routed (unnecessary API call):  {len(under)}")
if len(over):
    print("\nover-routed prompts:")
    print(over[["state", "prompt_id", "message"]].drop_duplicates().to_string(index=False))
if len(under):
    print("\nunder-routed prompts:")
    print(under[["state", "prompt_id", "message"]].drop_duplicates().to_string(index=False))

nearmiss = base[base["prompt_id"].str.startswith("nearmiss_")]
if not nearmiss.empty:
    nm = (nearmiss.groupby("prompt_id")
          .agg(fell_through=("actual_route", lambda s: (s != "button_route").mean()),
               n=("actual_route", "size")))
    print("\n[A] near-miss prompts — 1.0 means correctly fell through to Claude")
    print(nm.round(3))

## 6. Retrieval (A)

**Only BM25 is implemented.** The cosine and RRF arms of the original plan
were deferred pending a decision on an embedding source, so this section
shows one ranking, not three fused. Saying so is the point: a fusion chart
built from a ranker that does not exist would be the exact kind of number
this harness is supposed to make impossible.

The fixture plants three near-duplicate pairs. Each says the same thing
twice — once sharing the query's vocabulary, once paraphrased.

In [ ]:
import database  # noqa: E402
import retrieval  # noqa: E402

database.DB_PATH = EVAL_DIR / "db" / "full.db"
CHAT_ID = 999_000_001

PROBES = {
    "chain_rule": "chain rule gradient loss weights",
    "cache_miss": "cache line memory hierarchy stall",
    "admissible": "admissible heuristic never overestimates",
}

rows = []
for slug, query in PROBES.items():
    hits = database.search_notes(CHAT_ID, query, limit=10)
    ranked = {h["source"]: (i + 1, h["score"]) for i, h in enumerate(hits)}
    conn = database.get_connection()
    try:
        members = conn.execute(
            "SELECT id, content FROM notes WHERE source = ? ORDER BY id",
            (f"seed:{slug}",),
        ).fetchall()
    finally:
        conn.close()
    for position, (_, content) in enumerate(members):
        role = "lexical" if position == 0 else "paraphrase"
        rank, score = ranked.get(f"seed:{slug}", (None, None))
        # Both pair members share a source, so resolve by text instead.
        match = next(
            ((i + 1, h["score"]) for i, h in enumerate(hits)
             if h["text"][:40] in content),
            (None, 0.0),
        )
        rows.append({"pair": slug, "member": role,
                     "bm25_rank": match[0], "bm25_score": match[1]})

retr = pd.DataFrame(rows)
print("[A] BM25 on planted near-duplicate pairs "
      "(rank None = not retrieved at all)")
print(retr.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
pivot = retr.pivot(index="pair", columns="member", values="bm25_score").fillna(0)
pivot.plot(kind="bar", ax=ax, color=["#3b6ea5", "#c9a227"])
ax.set_ylabel("BM25 score")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.set_title("[A] BM25: lexical member vs paraphrase of the same fact")
fig.savefig(FIG_DIR / "05_retrieval_bm25.png", bbox_inches="tight")
plt.show()

missed = retr[(retr["member"] == "paraphrase") & (retr["bm25_rank"].isna())]
print(f"\n[A] paraphrases not retrieved at all: {len(missed)}/{len(PROBES)}")

## 7. Failure regression — dataset B only

This runs on real, hand-labelled turns and nothing else. Synthetic dispatch
accuracy is not a correctness label: the suite's expectations were written by
the same process that wrote the prompts, so fitting to them measures
agreement with my own guesses.

**Coefficients are reported; accuracy is not.** The sample is small and will
not support an accuracy claim. Coefficients are on standardized features, so
they are comparable to one another and are log-odds, not probabilities.

Labelling: add a `label` column to a copy of `turns` with values
`correct` / `soft_wrong` / `hard_fail`. `soft_wrong` is the class worth
predicting — a fluent reply that misreports what happened, which no automatic
signal catches.

In [ ]:
MIN_ROWS_FOR_REGRESSION = 40

if not HAS_REAL:
    print("[B] skipped — no real data yet.")
elif "label" not in real_turns.columns:
    print(f"[B] {len(real_turns)} turns collected, none labelled yet.\n"
          "    Add a `label` column (correct / soft_wrong / hard_fail) and re-run.")
else:
    labelled = real_turns[real_turns["label"].notna()].copy()
    print(f"[B] {len(labelled)} labelled turns of {len(real_turns)} collected")
    print(labelled["label"].value_counts().rename("turns"))

    if len(labelled) < MIN_ROWS_FOR_REGRESSION:
        print(f"\n[B] fewer than {MIN_ROWS_FOR_REGRESSION} labelled rows — "
              "not fitting. Coefficients from this little data would be noise "
              "with error bars wide enough to include zero in both directions.")
    else:
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler

        tool_mix = (
            real_calls.assign(one=1)
            .pivot_table(index=["conversation_id", "turn_index"],
                         columns="tool_name", values="one", aggfunc="sum")
            .fillna(0)
        )
        feats = (
            labelled.set_index(["conversation_id", "turn_index"])
            [["iterations", "message_chars", "api_calls"]]
            .join(tool_mix, how="left").fillna(0)
        )
        # Drop tools that are constant across the labelled set: a column with no
        # variance contributes no information and its coefficient is an artifact
        # of the regularizer, not a finding.
        feats = feats.loc[:, feats.nunique() > 1]
        y = (labelled.set_index(["conversation_id", "turn_index"])
             .loc[feats.index, "label"] != "correct").astype(int)

        X = StandardScaler().fit_transform(feats)
        model = LogisticRegression(max_iter=2000, class_weight="balanced")
        model.fit(X, y)

        coefs = (
            pd.DataFrame({"feature": feats.columns, "coef": model.coef_[0]})
            .assign(abs_coef=lambda d: d["coef"].abs())
            .sort_values("abs_coef", ascending=False).drop(columns="abs_coef")
        )
        print(f"\n[B] logistic regression, n={len(y)}, "
              f"{int(y.sum())} failures / {int((~y.astype(bool)).sum())} correct")
        print("    standardized coefficients (log-odds of failure)")
        print(coefs.round(3).to_string(index=False))
        print("\n    No accuracy or AUC is reported. At this n it would be a "
              "number about the split, not about the bot.")

        fig, ax = plt.subplots(figsize=(7, 5))
        show = coefs.head(12).iloc[::-1]
        ax.barh(show["feature"], show["coef"],
                color=["#c25b56" if c > 0 else "#3b6ea5" for c in show["coef"]])
        ax.axvline(0, color="#444", lw=1)
        ax.set_xlabel("standardized coefficient (log-odds of failure)")
        ax.set_title(f"[B] Failure predictors, n={len(y)} labelled real turns")
        fig.savefig(FIG_DIR / "06_failure_coefficients.png", bbox_inches="tight")
        plt.show()

## Sample sizes

Stated separately, never summed.

In [ ]:
summary = pd.DataFrame([
    {"dataset": "A synthetic", "unit": "executions", "n": len(syn_exec)},
    {"dataset": "A synthetic", "unit": "tool calls", "n": len(syn_calls)},
    {"dataset": "A synthetic", "unit": "distinct prompts",
     "n": syn_exec["prompt_id"].nunique()},
    {"dataset": "B real", "unit": "turns",
     "n": len(real_turns) if HAS_REAL else 0},
    {"dataset": "B real", "unit": "tool calls",
     "n": len(real_calls) if HAS_REAL else 0},
])
print(summary.to_string(index=False))
print("\nfigures written to", FIG_DIR)
for path in sorted(FIG_DIR.glob("*.png")):
    print("  ", path.name)